# Transfer Learning — EfficientNetB0

## 1. Mount Google Drive & Import Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
import json
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB0
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU tersedia       : {tf.config.list_physical_devices("GPU")}')

## 2. Konfigurasi Path & Grid Hyperparameter

In [ ]:
# Path dataset
TRAIN_DIR      = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_GC_test/train'
VAL_DIR        = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_GC_test/val'
TEST_DIR       = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_GC_test/test'
TRAIN_DIR_ORIG = TRAIN_DIR

# Folder output (terpisah dari hasil CNN asli)
OUTPUT_DIR = '/content/drive/MyDrive/Tugas Akhir/TL Gradcam/TransferLearning_GC_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)
PROSES_DIR = os.path.join(OUTPUT_DIR, 'proses')
os.makedirs(PROSES_DIR, exist_ok=True)
HASIL_DIR  = os.path.join(PROSES_DIR, 'hasil')
os.makedirs(HASIL_DIR, exist_ok=True)
MODELS_DIR = os.path.join(PROSES_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Hyperparameter tetap
IMG_SIZE    = (224, 224)   # default EfficientNetB0
# Sesuai protokol training artikel (Section III.D.3 & Eq. 25): total epoch
# (Fase 1 + Fase 2) dibatasi maksimum E=50, bukan per-fase. FINETUNE_EPOCHS
# di bawah dihitung otomatis saat runtime = EPOCHS - epoch_terpakai_fase1.
EPOCHS      = 50           # E_max total sesuai artikel (Eq. 25), EarlyStopping bisa berhenti lebih awal
PATIENCE    = 10           # patience sesuai artikel (Section III.D.3), berlaku utk kedua fase
NUM_CLASSES = 2
CLASS_NAMES = ['no_tumor', 'tumor']

# Grid hyperparameter
PARAM_GRID = {
    'learning_rate': [0.0001, 0.001, 0.01],
    'batch_size'   : [16, 40, 64],
    'dropout_rate' : [0.1, 0.2, 0.3],
}

REDUCED_GRID = False  # True = 7 kombinasi cepat, False = 27 kombinasi penuh

FIXED_OPTIMIZER = 'adam'

REDUCED_COMBINATIONS = [
    (0.0001, 16,  0.1),
    (0.0001, 40,  0.1),
    (0.0001, 64,  0.1),
    (0.0001, 40,  0.2),
    (0.0001, 40,  0.3),
    (0.001,  40,  0.1),
    (0.01,   40,  0.1),
]

# Konfigurasi fine-tuning fase 2 (mengikuti Table 2 & Bagian III.B Deep-EFNet)
FINETUNE_UNFREEZE_LAYERS = 30    # jumlah layer terakhir backbone yang di-unfreeze (absolut, sesuai artikel)
FINETUNE_LR_FACTOR       = 10    # lr fase 2 = lr fase 1 x faktor ini (LEBIH BESAR, sesuai eta1=1e-4 -> eta2=1e-3)
# FINETUNE_EPOCHS TIDAK lagi konstan — dihitung dinamis per-run di loop grid search
# supaya epoch_fase1 + epoch_fase2 <= EPOCHS (total maksimum 50 sesuai artikel).

# Konfigurasi decay learning rate setelah T2 (Eq. 21, Bagian III.B.3):
# eta(t) = eta2 * gamma^floor((t-T2)/Td) untuk t > T2
LR_DECAY_GAMMA        = 0.1   # faktor decay sesuai artikel (gamma = 0.1)
LR_DECAY_T2_RATIO     = 0.5   # T2 = 50% dari total epoch fase 2 (mulai decay di tengah fine-tuning)
LR_DECAY_INTERVAL_TD  = 5     # Td: decay diterapkan tiap 5 epoch setelah T2

## 2.5 Augmentasi Data — 3 Metode (Iftikhar et al., 2025)

In [ ]:
import cv2
import glob
import random
import shutil

TRAIN_DIR_AUG = os.path.join(HASIL_DIR, 'dataset_augmented', 'train')
VAL_DIR_AUG   = VAL_DIR
TEST_DIR_AUG  = TEST_DIR

TARGET_MULTIPLIER = 4
AUG_PER_METHOD    = TARGET_MULTIPLIER - 1  # = 3
AUG_SEED = 42


def apply_shearing(img_rgb, shear_range=(-15, 15), rng=None):
    if rng is None: rng = random.Random()
    sx = np.tan(np.deg2rad(rng.uniform(*shear_range)))
    sy = np.tan(np.deg2rad(rng.uniform(*shear_range)))
    h, w = img_rgb.shape[:2]
    cx, cy = w/2.0, h/2.0
    M = np.float32([[1, sx, -cy*sx],[sy, 1, -cx*sy]])
    return cv2.warpAffine(img_rgb, M, (w,h), borderMode=cv2.BORDER_REFLECT_101)


def apply_rotation(img_rgb, angle_range=(-25, 25), rng=None):
    if rng is None: rng = random.Random()
    angle = rng.uniform(*angle_range)
    h, w = img_rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w/2.0, h/2.0), angle, 1.0)
    return cv2.warpAffine(img_rgb, M, (w,h), borderMode=cv2.BORDER_REFLECT_101)


def apply_translation(img_rgb, tx_range=(-15,15), ty_range=(-15,15), rng=None):
    if rng is None: rng = random.Random()
    M = np.float32([[1,0,rng.uniform(*tx_range)],[0,1,rng.uniform(*ty_range)]])
    h, w = img_rgb.shape[:2]
    return cv2.warpAffine(img_rgb, M, (w,h), borderMode=cv2.BORDER_REFLECT_101)


def apply_translation_shearing_rotation(img_rgb, tx_range=(-15,15), ty_range=(-15,15),
                                         shear_range=(-15,15), angle_range=(-25,25), rng=None):
    img = apply_translation(img_rgb, tx_range, ty_range, rng)
    img = apply_shearing(img, shear_range, rng)
    img = apply_rotation(img, angle_range, rng)
    return img


AUGMENTATION_METHODS = [
    ('shear',           apply_shearing,                      {}),
    ('rot',             apply_rotation,                      {}),
    ('trans_shear_rot', apply_translation_shearing_rotation, {}),
]


def run_augmentation(src_train_dir, dst_train_dir, class_names,
                     aug_per_method=3, img_size=(224,224), seed=42, overwrite=False):
    # Skip jika folder augmentasi sudah terisi
    if os.path.isdir(dst_train_dir) and not overwrite:
        existing = glob.glob(os.path.join(dst_train_dir, '**', '*.*'), recursive=True)
        if existing:
            return

    rng = random.Random(seed)
    n_methods = len(AUGMENTATION_METHODS)

    for cls_name in class_names:
        src_cls = os.path.join(src_train_dir, cls_name)
        dst_cls = os.path.join(dst_train_dir, cls_name)
        os.makedirs(dst_cls, exist_ok=True)

        img_paths = sorted(p for ext in ('*.png','*.jpg','*.jpeg')
                           for p in glob.glob(os.path.join(src_cls, ext)))
        if not img_paths:
            print(f'Peringatan: tidak ada gambar di {src_cls}')
            continue

        for img_path in img_paths:
            fname = os.path.basename(img_path)
            name, ext = os.path.splitext(fname)
            raw = cv2.imread(img_path)
            if raw is None: continue
            img_rgb = cv2.cvtColor(cv2.resize(raw, img_size), cv2.COLOR_BGR2RGB)
            cv2.imwrite(os.path.join(dst_cls, fname), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))

            # Terapkan metode augmentasi bergiliran
            for slot_idx in range(aug_per_method):
                method_suffix, aug_fn, kwargs = AUGMENTATION_METHODS[slot_idx % n_methods]
                aug_img = aug_fn(img_rgb, rng=rng, **kwargs)
                aug_fname = f'{name}_aug{slot_idx+1:02d}_{method_suffix}_s{slot_idx+1}{ext}'
                cv2.imwrite(os.path.join(dst_cls, aug_fname),
                            cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))


TRAIN_DIR_ORIG = TRAIN_DIR

## 2.6 Augmentasi & Persiapan Data Training

In [ ]:
def load_train_rgb_array(train_dir, class_names, resize_to=None):
    # float32 [0,255] — TIDAK dibagi 255, EfficientNetB0 punya preprocessing sendiri
    X_list, y_list = [], []
    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = os.path.join(train_dir, cls_name)
        if not os.path.isdir(cls_dir):
            print(f'Peringatan: folder tidak ditemukan: {cls_dir}')
            continue
        paths_cls = sorted(p for ext in ('*.png','*.jpg','*.jpeg')
                           for p in glob.glob(os.path.join(cls_dir, ext)))
        for fpath in paths_cls:
            raw = cv2.imread(fpath)
            if raw is None: continue
            img_rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
            if resize_to:
                img_rgb = cv2.resize(img_rgb, resize_to, interpolation=cv2.INTER_LINEAR)
            X_list.append(img_rgb.astype(np.float32))
            y_list.append(cls_idx)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)


X_orig_arr, y_orig_arr = load_train_rgb_array(TRAIN_DIR_ORIG, CLASS_NAMES, resize_to=IMG_SIZE)

run_augmentation(
    src_train_dir=TRAIN_DIR_ORIG, dst_train_dir=TRAIN_DIR_AUG,
    class_names=CLASS_NAMES, aug_per_method=AUG_PER_METHOD,
    img_size=IMG_SIZE, seed=AUG_SEED, overwrite=False,
)

X_train_final, y_train_final = load_train_rgb_array(TRAIN_DIR_AUG, CLASS_NAMES, resize_to=IMG_SIZE)

idx_shuf      = np.random.permutation(len(X_train_final))
X_train_final = X_train_final[idx_shuf]
y_train_final = y_train_final[idx_shuf]

print(f'Data training final: {len(y_train_final)} gambar, shape {X_train_final.shape}')

X_smote = X_train_final
y_smote = y_train_final
TRAIN_DIR = TRAIN_DIR_AUG

## 3. Fungsi Utilitas

In [ ]:
def make_generators(batch_size):
    # Tanpa rescale — EfficientNetB0 menangani preprocessing internal
    # class_mode='categorical' -> label one-hot, sesuai formulasi categorical
    # cross-entropy artikel (Eq. 14, z_j,d one-hot indicator)
    val_gen = ImageDataGenerator().flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', classes=CLASS_NAMES, shuffle=False
    )
    test_gen = ImageDataGenerator().flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', classes=CLASS_NAMES, shuffle=False
    )
    return val_gen, test_gen


def build_efficientnet(dropout_rate=0.3, num_classes=2, trainable_backbone=False):
    """
    Arsitektur head disesuaikan dengan Deep-EFNet (Alshaari & Alqahtani, 2025), Table 2:
      GAP -> BatchNorm -> Dropout -> Dense(512, ReLU, L2=0.01) -> BatchNorm -> Dropout
          -> Dense(256, ReLU, L2=0.01) -> BatchNorm -> Dropout -> Dense(softmax)
    Dropout rate TETAP mengikuti grid search (0.1-0.3), tidak di-hardcode 0.5 seperti artikel.
    """
    backbone = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3)
    )
    backbone.trainable = trainable_backbone

    l2_reg = keras.regularizers.l2(0.01)  # sesuai Eq. 22, lambda = 0.01

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='input_image')
    x = backbone(inputs, training=False)  # training=False agar BN backbone tetap inference mode
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='batchnorm_base')(x)
    x = layers.Dropout(dropout_rate, name='dropout0')(x)

    # Dense Layer 1 — 512 unit, ReLU, L2(0.01) (Table 2)
    x = layers.Dense(512, activation='relu', kernel_regularizer=l2_reg, name='dense1')(x)
    x = layers.BatchNormalization(name='batchnorm_dense1')(x)
    x = layers.Dropout(dropout_rate, name='dropout1')(x)

    # Dense Layer 2 — 256 unit, ReLU, L2(0.01) (Table 2)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2_reg, name='dense2')(x)
    x = layers.BatchNormalization(name='batchnorm_dense2')(x)
    x = layers.Dropout(dropout_rate, name='dropout2')(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='Deep_EFNet')
    return model, backbone


def unfreeze_top_layers(backbone, n_layers=30):
    """
    Sesuai artikel Bagian III.B.2 & Table 2: unfreeze n_layers (30) layer
    TERAKHIR backbone (angka absolut, bukan rasio/persentase). Tidak ada
    pengecualian khusus untuk BatchNormalization -- semua layer di 30 layer
    terakhir (termasuk BatchNorm) ikut dilatih, persis seperti dijelaskan
    di artikel (tidak disebutkan pengecualian apa pun untuk BatchNorm).
    """
    backbone.trainable = True
    total_layers = len(backbone.layers)
    freeze_until = max(0, total_layers - n_layers)

    for i, layer in enumerate(backbone.layers):
        layer.trainable = (i >= freeze_until)


def get_optimizer(name, lr):
    opts = {
        'adam'    : keras.optimizers.Adam(learning_rate=lr),
        'sgd'     : keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
        'rmsprop' : keras.optimizers.RMSprop(learning_rate=lr),
    }
    if name not in opts:
        raise ValueError(f'Optimizer tidak dikenal: {name}')
    return opts[name]


def combo_name(lr, bs, dr):
    return f'lr{lr}_bs{bs}_dr{dr}'

## 4. Generate Kombinasi Grid

In [ ]:
if REDUCED_GRID:
    combinations = REDUCED_COMBINATIONS
else:
    combinations = list(itertools.product(
        PARAM_GRID['learning_rate'],
        PARAM_GRID['batch_size'],
        PARAM_GRID['dropout_rate'],
    ))

total_runs = len(combinations)

df_grid = pd.DataFrame(combinations, columns=['learning_rate', 'batch_size', 'dropout_rate'])
df_grid.index = df_grid.index + 1
df_grid.index.name = 'Run'

## 5. Grid Search Loop Utama — 2 Fase Fine-Tuning

In [ ]:
all_results = []
done_names  = set()

In [ ]:
xgrid_start = time.time()

# Label one-hot untuk categorical cross-entropy (Eq. 14), dihitung sekali di luar loop
y_smote_ohe = tf.keras.utils.to_categorical(y_smote, num_classes=NUM_CLASSES)

for run_idx, (lr, bs, dr) in enumerate(combinations, start=1):
    name = combo_name(lr, bs, dr)
    if name in done_names:
        continue

    run_start = time.time()
    val_gen, test_gen = make_generators(bs)

    tf.keras.backend.clear_session()

    # Fase 1 -- Feature Extraction (backbone frozen)
    model, backbone = build_efficientnet(dropout_rate=dr, num_classes=NUM_CLASSES,
                                          trainable_backbone=False)
    model.compile(
        optimizer=get_optimizer(FIXED_OPTIMIZER, lr),
        loss='categorical_crossentropy',   # sesuai Eq. 14 (label one-hot)
        metrics=['accuracy']
    )

    ckpt_path = os.path.join(MODELS_DIR, f'{name}_phase1.keras')
    callbacks_p1 = [
        ModelCheckpoint(filepath=ckpt_path, monitor='val_loss',
                        save_best_only=True, mode='min', verbose=0),
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=0),
    ]

    hist_p1 = model.fit(
        X_smote, y_smote_ohe,
        epochs=EPOCHS,
        batch_size=bs,
        validation_data=val_gen,
        callbacks=callbacks_p1,
        verbose=0
    )
    epochs_p1 = len(hist_p1.history['val_loss'])

    # Fase 2 -- Fine-Tuning (unfreeze 30 layer atas, lr diperbesar sesuai artikel: eta1=1e-4 -> eta2=1e-3)
    lr_ft = lr * FINETUNE_LR_FACTOR
    unfreeze_top_layers(backbone, n_layers=FINETUNE_UNFREEZE_LAYERS)

    model.compile(
        optimizer=get_optimizer(FIXED_OPTIMIZER, lr_ft),
        loss='categorical_crossentropy',   # sesuai Eq. 14 (label one-hot)
        metrics=['accuracy']
    )

    # Total epoch (fase 1 + fase 2) dibatasi EPOCHS (=50) sesuai protokol artikel (Eq. 25)
    finetune_epochs_run = max(1, EPOCHS - epochs_p1)

    # Learning rate decay setelah T2 (Eq. 21): eta2 * gamma^floor((t-T2)/Td)
    t2_ft = max(1, int(finetune_epochs_run * LR_DECAY_T2_RATIO))

    def lr_schedule(epoch, current_lr, _lr_ft=lr_ft, _t2=t2_ft):
        if epoch <= _t2:
            return _lr_ft
        decay_steps = (epoch - _t2) // LR_DECAY_INTERVAL_TD
        return _lr_ft * (LR_DECAY_GAMMA ** decay_steps)

    ckpt_path_ft = os.path.join(MODELS_DIR, f'{name}.keras')
    callbacks_p2 = [
        ModelCheckpoint(filepath=ckpt_path_ft, monitor='val_loss',
                        save_best_only=True, mode='min', verbose=0),
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=0),
        LearningRateScheduler(lr_schedule, verbose=0),
    ]

    val_gen.reset()
    hist_p2 = model.fit(
        X_smote, y_smote_ohe,
        epochs=finetune_epochs_run,
        batch_size=bs,
        validation_data=val_gen,
        callbacks=callbacks_p2,
        verbose=0
    )
    epochs_p2 = len(hist_p2.history['val_loss'])

    combined_history = {
        'acc'     : hist_p1.history['accuracy']     + hist_p2.history['accuracy'],
        'val_acc' : hist_p1.history['val_accuracy'] + hist_p2.history['val_accuracy'],
        'loss'    : hist_p1.history['loss']         + hist_p2.history['loss'],
        'val_loss': hist_p1.history['val_loss']     + hist_p2.history['val_loss'],
        'phase1_end_epoch': epochs_p1,
    }

    # Evaluasi test set
    test_gen.reset()
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)

    test_gen.reset()
    y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
    y_true = test_gen.classes

    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                    output_dict=True)
    f1_weighted  = report['weighted avg']['f1-score']
    precision_w  = report['weighted avg']['precision']
    recall_w     = report['weighted avg']['recall']

    cm_run = confusion_matrix(y_true, y_pred)
    tn_run = cm_run[0, 0]
    fp_run = cm_run[0, 1]
    specificity = tn_run / (tn_run + fp_run) if (tn_run + fp_run) > 0 else 0.0

    best_val_loss  = min(combined_history['val_loss'])
    best_val_acc   = max(combined_history['val_acc'])
    epochs_total   = epochs_p1 + epochs_p2
    run_time = time.time() - run_start

    result = {
        'run'              : run_idx,
        'name'             : name,
        'learning_rate'    : lr,
        'batch_size'       : bs,
        'dropout_rate'     : dr,
        'epochs_phase1'    : epochs_p1,
        'epochs_phase2'    : epochs_p2,
        'epochs_total'     : epochs_total,
        'best_val_loss'    : round(best_val_loss, 6),
        'best_val_acc'     : round(best_val_acc,  6),
        'test_loss'        : round(test_loss, 6),
        'test_acc'         : round(test_acc,  6),
        'f1_weighted'      : round(f1_weighted, 6),
        'precision_w'      : round(precision_w,  6),
        'recall_w'         : round(recall_w,     6),
        'specificity'      : round(specificity,  6),
        'optimizer'        : FIXED_OPTIMIZER,
        'run_time_sec'     : round(run_time, 1),
        'history_val_loss' : combined_history['val_loss'],
        'history_val_acc'  : combined_history['val_acc'],
        'history_loss'     : combined_history['loss'],
        'history_acc'      : combined_history['acc'],
        'phase1_end_epoch' : epochs_p1,
    }

    all_results.append(result)
    done_names.add(name)

    print(f'[{run_idx:02d}/{total_runs}] {name}: test_acc={test_acc:.4f}, f1={f1_weighted:.4f}')

total_time = (time.time() - xgrid_start) / 60
print(f'Grid Search selesai. Total waktu: {total_time:.1f} menit')


## 6. Tabel Ringkasan Hasil

In [ ]:
cols = ['run', 'learning_rate', 'batch_size', 'dropout_rate',
        'epochs_phase1', 'epochs_phase2', 'epochs_total',
        'best_val_acc', 'test_acc', 'f1_weighted',
        'precision_w', 'recall_w', 'specificity',
        'best_val_loss', 'test_loss', 'run_time_sec']

df_results = pd.DataFrame([{c: r.get(c, float('nan')) for c in cols} for r in all_results])

# Composite score — bobot: accuracy, precision, recall, specificity, F1
W_ACCURACY    = 0.20
W_PRECISION   = 0.15
W_RECALL      = 0.30
W_SPECIFICITY = 0.15
W_F1          = 0.20

df_results['composite_score'] = (
    W_ACCURACY    * df_results['test_acc']     +
    W_PRECISION   * df_results['precision_w']  +
    W_RECALL      * df_results['recall_w']     +
    W_SPECIFICITY * df_results['specificity']  +
    W_F1          * df_results['f1_weighted']
)

df_results = df_results.sort_values('composite_score', ascending=False).reset_index(drop=True)

df_results['test_acc_%']     = (df_results['test_acc']        * 100).round(2)
df_results['best_val_acc_%'] = (df_results['best_val_acc']    * 100).round(2)
df_results['f1_%']           = (df_results['f1_weighted']     * 100).round(2)
df_results['precision_%']    = (df_results['precision_w']     * 100).round(2)
df_results['recall_%']       = (df_results['recall_w']        * 100).round(2)
df_results['specificity_%']  = (df_results['specificity']     * 100).round(2)
df_results['composite_%']    = (df_results['composite_score'] * 100).round(2)
df_results['time_min']       = (df_results['run_time_sec']    / 60).round(1)

display_cols_full = [
    'run', 'learning_rate', 'batch_size', 'dropout_rate',
    'test_acc_%', 'precision_%', 'recall_%', 'specificity_%', 'f1_%',
    'composite_%', 'epochs_total', 'time_min'
]

display(df_results[display_cols_full].rename(columns={
    'learning_rate' : 'LR',
    'batch_size'    : 'BS',
    'dropout_rate'  : 'Dropout',
    'test_acc_%'    : 'Accuracy (%)',
    'precision_%'   : 'Precision (%)',
    'recall_%'      : 'Recall (%)',
    'specificity_%' : 'Specificity (%)',
    'f1_%'          : 'F1 (%)',
    'composite_%'   : 'Composite Score (%)',
    'epochs_total'  : 'Total Epoch',
    'time_min'      : 'Waktu (mnt)',
}))

best = df_results.iloc[0]
print(f'Kombinasi terbaik: lr={best.learning_rate}, bs={best.batch_size}, dropout={best.dropout_rate}')
print(f'Accuracy={best["test_acc_%"]:.2f}% | Precision={best["precision_%"]:.2f}% | '
      f'Recall={best["recall_%"]:.2f}% | Specificity={best["specificity_%"]:.2f}% | '
      f'F1={best["f1_%"]:.2f}% | Composite={best["composite_%"]:.2f}%')

## 7. Simpan Tabel ke CSV

In [ ]:
csv_path = os.path.join(PROSES_DIR, 'tl_grid_results_summary.csv')
df_results[display_cols_full].to_csv(csv_path, index=False)
print(f'Tabel disimpan ke: {csv_path}')

## 11. Learning Curves — Top-3 Kombinasi

In [ ]:
top3_runs  = df_results.head(3)['run'].tolist()
top3_res   = [r for r in all_results if r['run'] in top3_runs]
name_order = {r['run']: i for i, r in enumerate(df_results.head(3).to_dict('records'))}
top3_res.sort(key=lambda r: name_order.get(r['run'], 99))

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Learning Curves — Top-3 Kombinasi Transfer Learning (EfficientNetB0)\n'
             '(garis vertikal = batas Fase 1 → Fase 2 Fine-Tuning)',
             fontsize=13, fontweight='bold')

for row, r in enumerate(top3_res):
    epochs_x = range(1, len(r['history_val_loss']) + 1)
    p1_end   = r.get('phase1_end_epoch', None)
    title = (f'#{row+1}: lr={r["learning_rate"]} | bs={r["batch_size"]} | '
             f'dr={r["dropout_rate"]}\n'
             f'Test Acc: {r["test_acc"]*100:.2f}% | F1: {r["f1_weighted"]*100:.2f}%')

    # Accuracy
    axes[row,0].plot(epochs_x, r['history_acc'],     label='Train', color='royalblue')
    axes[row,0].plot(epochs_x, r['history_val_acc'], label='Val',   color='tomato', linestyle='--')
    if p1_end:
        axes[row,0].axvline(x=p1_end+0.5, color='gray', linestyle=':', linewidth=1.5,
                            label=f'Fine-tune start (ep {p1_end+1})')
    axes[row,0].set_title(title, fontsize=9, fontweight='bold')
    axes[row,0].set_ylabel('Accuracy')
    axes[row,0].legend(fontsize=8)
    axes[row,0].grid(alpha=0.3)

    # Loss
    axes[row,1].plot(epochs_x, r['history_loss'],     label='Train', color='royalblue')
    axes[row,1].plot(epochs_x, r['history_val_loss'], label='Val',   color='tomato', linestyle='--')
    if p1_end:
        axes[row,1].axvline(x=p1_end+0.5, color='gray', linestyle=':', linewidth=1.5)
    axes[row,1].set_ylabel('Loss')
    axes[row,1].legend(fontsize=8)
    axes[row,1].grid(alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(PROSES_DIR, 'tl_top3_curves.png'), dpi=150)
plt.show()

## 12. Confusion Matrix — Kombinasi Terbaik

In [ ]:
best_row = df_results.iloc[0]
best_lr  = best_row['learning_rate']
best_bs  = int(best_row['batch_size'])
best_dr  = best_row['dropout_rate']

best_ckpt = os.path.join(MODELS_DIR, f'{combo_name(best_lr, best_bs, best_dr)}.keras')
tf.keras.backend.clear_session()
best_model = keras.models.load_model(best_ckpt)

_, test_gen = make_generators(best_bs)
test_gen.reset()
y_pred = np.argmax(best_model.predict(test_gen, verbose=0), axis=1)
y_true = test_gen.classes

print('Classification Report — Transfer Learning Terbaik')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Confusion Matrix — EfficientNetB0 Terbaik\n'
             f'lr={best_lr} | bs={best_bs} | dropout={best_dr}', fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(PROSES_DIR, 'tl_best_confusion_matrix.png'), dpi=150)
plt.show()

## 13. Simpan Model Terbaik

In [ ]:
final_path = os.path.join(PROSES_DIR, 'best_model_tl_efficientnet.keras')
best_model.save(final_path)
print(f'Model terbaik disimpan ke: {final_path}')

best_config = {
    'backbone'       : 'EfficientNetB0',
    'learning_rate'  : best_lr,
    'batch_size'     : best_bs,
    'optimizer'      : FIXED_OPTIMIZER,
    'dropout_rate'   : best_dr,
    'epochs_phase1'  : int(best_row['epochs_phase1']),
    'epochs_phase2'  : int(best_row['epochs_phase2']),
    'finetune_unfreeze_layers' : FINETUNE_UNFREEZE_LAYERS,
    'finetune_lr_factor'       : FINETUNE_LR_FACTOR,
    'test_acc'       : float(best_row['test_acc']),
    'f1_weighted'    : float(best_row['f1_weighted']),
    'precision_w'    : float(best_row['precision_w']),
    'recall_w'       : float(best_row['recall_w']),
    'specificity'    : float(best_row['specificity']),
    'composite_score': float(best_row['composite_score']),
}
with open(os.path.join(PROSES_DIR, 'best_config_tl.json'), 'w') as f:
    json.dump(best_config, f, indent=2)

## 14. Select-Shuffle-Test — 10-Fold CV

In [ ]:
import os
import gc
import json
import glob as _glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# Muat hyperparameter terbaik dari hasil grid search
best_config_path = os.path.join(PROSES_DIR, 'best_config_tl.json')
with open(best_config_path, 'r') as f:
    best_config_cv = json.load(f)

BEST_LR  = best_config_cv['learning_rate']
BEST_BS  = int(best_config_cv['batch_size'])
BEST_OPT = best_config_cv['optimizer']
BEST_DR  = best_config_cv['dropout_rate']


# Ekstraksi ID pasien dari nama file, untuk group-aware split (cegah data leakage)
CT_FILENAME_PATTERN = re.compile(r'^(?P<patient_id>.+)_CT_[sm]\d+$')


def extract_patient_id(fpath: str) -> str:
    stem = Path(fpath).stem
    if '__' not in stem:
        return stem
    src_folder, orig_stem = stem.split('__', 1)
    if src_folder == 'image_tumor_mask':
        m = CT_FILENAME_PATTERN.match(orig_stem)
        pid = m.group('patient_id') if m else orig_stem
    else:
        part = orig_stem.split('_')[0]
        pid = part if part.isdigit() else orig_stem
    return f'{src_folder}__{pid}'


# Kumpulkan path + label + group TANPA load gambar ke RAM (dibaca on-demand via tf.data)
def collect_all_paths(dirs, class_names):
    all_paths, all_labels, all_groups = [], [], []
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    for d in dirs:
        for cls_name, cls_idx in class_to_idx.items():
            cls_dir = os.path.join(d, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            for ext in ('*.png', '*.jpg', '*.jpeg'):
                for fpath in _glob.glob(os.path.join(cls_dir, ext)):
                    if os.path.exists(fpath):
                        all_paths.append(fpath)
                        all_labels.append(cls_idx)
                        all_groups.append(extract_patient_id(fpath))
    return all_paths, np.array(all_labels), np.array(all_groups)


all_paths, all_labels, all_groups = collect_all_paths(
    dirs=[TRAIN_DIR_ORIG, VAL_DIR, TEST_DIR], class_names=CLASS_NAMES)


def make_tf_dataset(paths, labels, img_size, batch_size, training=False,
                    shuffle_buffer=1000):
    # Baca gambar dari disk per-batch (bukan load semua ke RAM), decode di dalam TF graph
    path_tensor  = tf.constant(paths,            dtype=tf.string)
    label_tensor = tf.constant(labels.tolist(),  dtype=tf.int32)

    def _parse(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)   # [0,255] — EfficientNet preprocess sendiri
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((path_tensor, label_tensor))
    if training:
        ds = ds.shuffle(buffer_size=shuffle_buffer, seed=42, reshuffle_each_iteration=True)
    ds = (ds
          .map(_parse, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(batch_size)
          .prefetch(tf.data.AUTOTUNE))
    return ds


# Shuffle berbasis index karena path disimpan sebagai list, bukan array gambar
SHUFFLE_SEED = 2025
rng_shuffle  = np.random.default_rng(SHUFFLE_SEED)
shuffle_idx  = rng_shuffle.permutation(len(all_paths))

all_paths_s  = [all_paths[i]  for i in shuffle_idx]
all_labels_s = all_labels[shuffle_idx]
all_groups_s = all_groups[shuffle_idx]

K_FOLDS    = 10
CV_EPOCHS  = 40
CV_DIR     = os.path.join(PROSES_DIR, 'CV_Results')
os.makedirs(CV_DIR, exist_ok=True)

for idx, cls in enumerate(CLASS_NAMES):
    n_pat = len(np.unique(all_groups_s[all_labels_s == idx]))
    if n_pat < K_FOLDS:
        print(f'Peringatan: kelas "{cls}" hanya punya {n_pat} pasien (< K_FOLDS={K_FOLDS})')

skf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=False)

cv_records   = []
cv_histories = []

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(all_paths_s, all_labels_s, groups=all_groups_s), start=1):

    # Sanity check: tidak boleh ada pasien yang sama di train & val
    train_patients = set(all_groups_s[train_idx])
    val_patients   = set(all_groups_s[val_idx])
    overlap        = train_patients & val_patients
    assert len(overlap) == 0, (
        f'[Fold {fold_idx}] LEAKAGE TERDETEKSI: {len(overlap)} pasien '
        f'muncul di train DAN val: {sorted(overlap)[:5]}...'
    )

    train_paths_fold = [all_paths_s[i] for i in train_idx]
    val_paths_fold   = [all_paths_s[i] for i in val_idx]
    y_tr  = all_labels_s[train_idx]
    y_val = all_labels_s[val_idx]

    ds_train = make_tf_dataset(train_paths_fold, y_tr,
                               IMG_SIZE, BEST_BS, training=True)
    ds_val   = make_tf_dataset(val_paths_fold,   y_val,
                               IMG_SIZE, BEST_BS, training=False)

    cw      = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_tr)
    cw_dict = dict(enumerate(cw))

    tf.keras.backend.clear_session()

    # Fase 1 — backbone beku
    model_cv, backbone_cv = build_efficientnet(
        dropout_rate=BEST_DR, num_classes=NUM_CLASSES, trainable_backbone=False)
    model_cv.compile(
        optimizer=get_optimizer(BEST_OPT, BEST_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])

    ckpt_cv_p1 = os.path.join(CV_DIR, f'fold_{fold_idx:02d}_p1.keras')
    hist_p1_cv = model_cv.fit(
        ds_train,
        epochs=CV_EPOCHS,
        validation_data=ds_val,
        class_weight=cw_dict,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                ckpt_cv_p1, monitor='val_loss',
                save_best_only=True, mode='min', verbose=0),
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=7,
                restore_best_weights=True, verbose=0),
        ],
        verbose=0
    )
    ep_p1 = len(hist_p1_cv.history['val_loss'])

    # Fase 2 — fine-tuning
    unfreeze_top_layers(backbone_cv, n_layers=FINETUNE_UNFREEZE_LAYERS)
    model_cv.compile(
        optimizer=get_optimizer(BEST_OPT, BEST_LR * FINETUNE_LR_FACTOR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])

    ckpt_cv_ft = os.path.join(CV_DIR, f'fold_{fold_idx:02d}.keras')
    hist_ft_cv = model_cv.fit(
        ds_train,
        epochs=FINETUNE_EPOCHS,
        validation_data=ds_val,
        class_weight=cw_dict,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                ckpt_cv_ft, monitor='val_loss',
                save_best_only=True, mode='min', verbose=0),
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=7,
                restore_best_weights=True, verbose=0),
        ],
        verbose=0
    )

    combined_h = {
        'accuracy':     hist_p1_cv.history['accuracy']     + hist_ft_cv.history['accuracy'],
        'val_accuracy': hist_p1_cv.history['val_accuracy'] + hist_ft_cv.history['val_accuracy'],
        'loss':         hist_p1_cv.history['loss']         + hist_ft_cv.history['loss'],
        'val_loss':     hist_p1_cv.history['val_loss']     + hist_ft_cv.history['val_loss'],
    }
    cv_histories.append(combined_h)

    val_loss, val_acc = model_cv.evaluate(ds_val, verbose=0)

    y_pred_list, y_true_list = [], []
    for batch_imgs, batch_labels in ds_val:
        preds = model_cv.predict_on_batch(batch_imgs)
        y_pred_list.extend(np.argmax(preds, axis=1).tolist())
        y_true_list.extend(batch_labels.numpy().tolist())
    y_pred_cv = np.array(y_pred_list)
    y_true_cv = np.array(y_true_list)

    report_cv = classification_report(
        y_true_cv, y_pred_cv,
        target_names=CLASS_NAMES,
        output_dict=True, zero_division=0)

    fold_result = {
        'fold':             fold_idx,
        'n_patients_train': len(train_patients),
        'n_patients_val':   len(val_patients),
        'val_loss':         round(val_loss, 4),
        'val_acc':          round(val_acc, 4),
        'f1_weighted':      round(report_cv['weighted avg']['f1-score'], 4),
        'precision_w':      round(report_cv['weighted avg']['precision'], 4),
        'recall_w':         round(report_cv['weighted avg']['recall'], 4),
        'f1_no_tumor':      round(report_cv['no_tumor']['f1-score'], 4),
        'f1_tumor':         round(report_cv['tumor']['f1-score'], 4),
        'epochs_run':       ep_p1 + len(hist_ft_cv.history['val_loss']),
        'best_val_loss':    round(min(combined_h['val_loss']), 4),
    }
    cv_records.append(fold_result)

    print(f'[Fold {fold_idx:02d}/{K_FOLDS}] acc={val_acc*100:.2f}% f1={fold_result["f1_weighted"]*100:.2f}%')

    # Bebaskan memori GPU eksplisit tiap akhir fold: del dulu baru clear_session + gc
    # agar VRAM benar-benar terlepas (cegah OOM kumulatif antar fold)
    del model_cv, backbone_cv
    del ds_train, ds_val
    del train_paths_fold, val_paths_fold
    del y_tr, y_val, y_pred_cv, y_true_cv
    del hist_p1_cv, hist_ft_cv

    tf.keras.backend.clear_session()
    gc.collect()


# Ringkasan statistik
df_cv = pd.DataFrame(cv_records).set_index('fold')

metrics_summary = ['val_acc', 'f1_weighted', 'precision_w', 'recall_w',
                   'f1_no_tumor', 'f1_tumor', 'val_loss']
summary_rows = []
for m in metrics_summary:
    vals = df_cv[m].values
    summary_rows.append({
        'Metric': m,
        'Mean':   round(vals.mean(), 4),
        'Std':    round(vals.std(),  4),
        'Min':    round(vals.min(),  4),
        'Max':    round(vals.max(),  4),
    })
df_summary_cv = pd.DataFrame(summary_rows).set_index('Metric')

mean_acc = df_cv['val_acc'].mean()
std_acc  = df_cv['val_acc'].std()
mean_f1  = df_cv['f1_weighted'].mean()
std_f1   = df_cv['f1_weighted'].std()

print(f'10-Fold CV selesai. Accuracy={mean_acc*100:.2f}%+/-{std_acc*100:.2f}% | '
      f'F1={mean_f1*100:.2f}%+/-{std_f1*100:.2f}%')

df_cv.to_csv(os.path.join(CV_DIR, 'tl_cv_fold_results.csv'))
df_summary_cv.to_csv(os.path.join(CV_DIR, 'tl_cv_summary.csv'))


# Plot CV
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle(
    f'Select-Shuffle-Test — 10-Fold CV — Transfer Learning EfficientNetB0\n'
    f'lr={BEST_LR} | bs={BEST_BS} | {BEST_OPT} | dropout={BEST_DR}',
    fontsize=13, fontweight='bold')

folds_x = df_cv.index.tolist()

ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(folds_x, df_cv['val_acc'] * 100,
        color='#3498db', alpha=0.85, edgecolor='white')
ax1.axhline(mean_acc * 100, color='red', linestyle='--', linewidth=1.5,
            label=f'Mean={mean_acc*100:.2f}%')
ax1.fill_between(folds_x,
                 (mean_acc - std_acc) * 100,
                 (mean_acc + std_acc) * 100,
                 alpha=0.15, color='red', label=f'+/-{std_acc*100:.2f}%')
ax1.set_title('Accuracy per Fold', fontweight='bold')
ax1.set_xlabel('Fold'); ax1.set_ylabel('Accuracy (%)')
ax1.set_xticks(folds_x); ax1.legend(fontsize=8); ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(folds_x, df_cv['f1_weighted'] * 100,
        color='#2ecc71', alpha=0.85, edgecolor='white')
ax2.axhline(mean_f1 * 100, color='red', linestyle='--', linewidth=1.5,
            label=f'Mean={mean_f1*100:.2f}%')
ax2.fill_between(folds_x,
                 (mean_f1 - std_f1) * 100,
                 (mean_f1 + std_f1) * 100,
                 alpha=0.15, color='red')
ax2.set_title('F1-Score per Fold', fontweight='bold')
ax2.set_xlabel('Fold'); ax2.set_ylabel('F1 (%)')
ax2.set_xticks(folds_x); ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
box_data = [df_cv['val_acc']      * 100,
            df_cv['f1_weighted']  * 100,
            df_cv['precision_w']  * 100,
            df_cv['recall_w']     * 100]
bp = ax3.boxplot(
    box_data,
    labels=['Accuracy', 'F1\n(weighted)', 'Precision\n(w)', 'Recall\n(w)'],
    patch_artist=True,
    medianprops=dict(color='red', linewidth=2))
for patch, c in zip(bp['boxes'], ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax3.set_title('Distribusi Metrik (10 Fold)', fontweight='bold')
ax3.set_ylabel('Score (%)'); ax3.grid(axis='y', alpha=0.3)

min_len     = min(len(h['val_loss']) for h in cv_histories)
acc_mat     = np.array([h['accuracy']     [:min_len] for h in cv_histories])
val_acc_mat = np.array([h['val_accuracy'] [:min_len] for h in cv_histories])
loss_mat    = np.array([h['loss']         [:min_len] for h in cv_histories])
val_loss_mat= np.array([h['val_loss']     [:min_len] for h in cv_histories])
ep_x        = range(1, min_len + 1)

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.plot(ep_x, acc_mat.mean(0)     * 100,
         color='royalblue', label='Train Acc (mean)')
ax4.plot(ep_x, val_acc_mat.mean(0) * 100,
         color='tomato',    label='Val Acc (mean)', linestyle='--')
ax4.fill_between(ep_x,
                 (val_acc_mat.mean(0) - val_acc_mat.std(0)) * 100,
                 (val_acc_mat.mean(0) + val_acc_mat.std(0)) * 100,
                 alpha=0.15, color='tomato')
ax4.set_title('Learning Curve Rata-Rata (10 Fold) — Accuracy', fontweight='bold')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Accuracy (%)'); ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(ep_x, loss_mat.mean(0),     color='royalblue', label='Train Loss')
ax5.plot(ep_x, val_loss_mat.mean(0), color='tomato',    label='Val Loss', linestyle='--')
ax5.fill_between(ep_x,
                 val_loss_mat.mean(0) - val_loss_mat.std(0),
                 val_loss_mat.mean(0) + val_loss_mat.std(0),
                 alpha=0.15, color='tomato')
ax5.set_title('Learning Curve Rata-Rata\n(10 Fold) — Loss', fontweight='bold')
ax5.set_xlabel('Epoch'); ax5.set_ylabel('Loss'); ax5.legend(fontsize=8)
ax5.grid(alpha=0.3)

plt.savefig(os.path.join(CV_DIR, 'tl_cv_plots.png'), dpi=150, bbox_inches='tight')
plt.show()


# Perbandingan Grid Search vs CV
print(f'Grid Search test_acc={best_config_cv["test_acc"]*100:.2f}% vs '
      f'10-Fold CV acc={mean_acc*100:.2f}%+/-{std_acc*100:.2f}%')
print(f'Grid Search F1={best_config_cv["f1_weighted"]*100:.2f}% vs '
      f'10-Fold CV F1={mean_f1*100:.2f}%+/-{std_f1*100:.2f}%')

In [ ]:
df_cv = pd.DataFrame(cv_records).set_index('fold')

metrics_summary = ['val_acc','f1_weighted','precision_w','recall_w',
                   'f1_no_tumor','f1_tumor','val_loss']
summary_rows = []
for m in metrics_summary:
    vals = df_cv[m].values
    summary_rows.append({'Metric': m, 'Mean': round(vals.mean(),4),
                         'Std': round(vals.std(),4), 'Min': round(vals.min(),4),
                         'Max': round(vals.max(),4)})
df_summary_cv = pd.DataFrame(summary_rows).set_index('Metric')

mean_acc = df_cv['val_acc'].mean()
std_acc  = df_cv['val_acc'].std()
mean_f1  = df_cv['f1_weighted'].mean()
std_f1   = df_cv['f1_weighted'].std()

print(f'10-Fold CV: Accuracy={mean_acc*100:.2f}%+/-{std_acc*100:.2f}% | '
      f'F1={mean_f1*100:.2f}%+/-{std_f1*100:.2f}%')

df_cv.to_csv(os.path.join(CV_DIR, 'tl_cv_fold_results.csv'))
df_summary_cv.to_csv(os.path.join(CV_DIR, 'tl_cv_summary.csv'))

In [ ]:
# Plot CV
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle(
    f'Select-Shuffle-Test — 10-Fold CV — Transfer Learning EfficientNetB0\n'
    f'lr={BEST_LR} | bs={BEST_BS} | {BEST_OPT} | dropout={BEST_DR}',
    fontsize=13, fontweight='bold')

folds_x = df_cv.index.tolist()

ax1 = fig.add_subplot(gs[0,0])
ax1.bar(folds_x, df_cv['val_acc']*100, color='#3498db', alpha=0.85, edgecolor='white')
ax1.axhline(mean_acc*100, color='red', linestyle='--', linewidth=1.5,
            label=f'Mean={mean_acc*100:.2f}%')
ax1.fill_between(folds_x, (mean_acc-std_acc)*100, (mean_acc+std_acc)*100,
                 alpha=0.15, color='red', label=f'+/-{std_acc*100:.2f}%')
ax1.set_title('Accuracy per Fold', fontweight='bold')
ax1.set_xlabel('Fold'); ax1.set_ylabel('Accuracy (%)')
ax1.set_xticks(folds_x); ax1.legend(fontsize=8); ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0,1])
ax2.bar(folds_x, df_cv['f1_weighted']*100, color='#2ecc71', alpha=0.85, edgecolor='white')
ax2.axhline(mean_f1*100, color='red', linestyle='--', linewidth=1.5,
            label=f'Mean={mean_f1*100:.2f}%')
ax2.fill_between(folds_x, (mean_f1-std_f1)*100, (mean_f1+std_f1)*100,
                 alpha=0.15, color='red')
ax2.set_title('F1-Score per Fold', fontweight='bold')
ax2.set_xlabel('Fold'); ax2.set_ylabel('F1 (%)')
ax2.set_xticks(folds_x); ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[0,2])
box_data  = [df_cv['val_acc']*100, df_cv['f1_weighted']*100,
             df_cv['precision_w']*100, df_cv['recall_w']*100]
bp = ax3.boxplot(box_data, labels=['Accuracy','F1\n(weighted)','Precision\n(w)','Recall\n(w)'],
                 patch_artist=True, medianprops=dict(color='red', linewidth=2))
for patch, c in zip(bp['boxes'], ['#3498db','#2ecc71','#e67e22','#9b59b6']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax3.set_title('Distribusi Metrik (10 Fold)', fontweight='bold')
ax3.set_ylabel('Score (%)'); ax3.grid(axis='y', alpha=0.3)

min_len     = min(len(h['val_loss']) for h in cv_histories)
acc_mat     = np.array([h['accuracy']    [:min_len] for h in cv_histories])
val_acc_mat = np.array([h['val_accuracy'][:min_len] for h in cv_histories])
loss_mat    = np.array([h['loss']        [:min_len] for h in cv_histories])
val_loss_mat= np.array([h['val_loss']    [:min_len] for h in cv_histories])
ep_x = range(1, min_len+1)

ax4 = fig.add_subplot(gs[1,0:2])
ax4.plot(ep_x, acc_mat.mean(0)*100,     color='royalblue', label='Train Acc (mean)')
ax4.plot(ep_x, val_acc_mat.mean(0)*100, color='tomato',    label='Val Acc (mean)', linestyle='--')
ax4.fill_between(ep_x,
    (val_acc_mat.mean(0)-val_acc_mat.std(0))*100,
    (val_acc_mat.mean(0)+val_acc_mat.std(0))*100,
    alpha=0.15, color='tomato')
ax4.set_title('Learning Curve Rata-Rata (10 Fold) — Accuracy', fontweight='bold')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Accuracy (%)'); ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

ax5 = fig.add_subplot(gs[1,2])
ax5.plot(ep_x, loss_mat.mean(0),     color='royalblue', label='Train Loss')
ax5.plot(ep_x, val_loss_mat.mean(0), color='tomato',    label='Val Loss', linestyle='--')
ax5.fill_between(ep_x,
    val_loss_mat.mean(0)-val_loss_mat.std(0),
    val_loss_mat.mean(0)+val_loss_mat.std(0),
    alpha=0.15, color='tomato')
ax5.set_title('Learning Curve Rata-Rata\n(10 Fold) — Loss', fontweight='bold')
ax5.set_xlabel('Epoch'); ax5.set_ylabel('Loss'); ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

plt.savefig(os.path.join(CV_DIR, 'tl_cv_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

# Perbandingan Grid Search vs CV
print(f'Grid Search test_acc={best_config_cv["test_acc"]*100:.2f}% vs '
      f'10-Fold CV acc={mean_acc*100:.2f}%+/-{std_acc*100:.2f}%')
print(f'Grid Search F1={best_config_cv["f1_weighted"]*100:.2f}% vs '
      f'10-Fold CV F1={mean_f1*100:.2f}%+/-{std_f1*100:.2f}%')

## Ringkasan Notebook Transfer Learning